# The Japanese classification table (TLS222_APPLN_JP_CLASS)

Welcome to the Japanese classification table in PATSTAT, namely table TLS222_APPLN_JP_CLASS. Here we can find stored the FI and F-terms linked to JP applications only. The Japanese classification schemes FI and FTERM are used by the Japan Patent Office for carrying out patent application searches. The FI scheme is built on top of the International Patent Classification system (IPC) and is constantly being revised and updated. The FTERM scheme contains technical terms attributed from multiple viewpoints to facilitate computerised retrieval of patent documents.

**FI** (File Index) has been developed to expand IPC in some technical fields. FI consists of an IPC symbol and an IPC-subdivision symbol and/or file discrimination symbol added to the IPC symbol.

**F-TERMS** (file forming terms) reclassify or further segment each specific technical field of IPC from a variety of viewpoints.

In [1]:
from epo.tipdata.patstat import PatstatClient

# Initialize the PATSTAT client
patstat = PatstatClient(env='PROD')

# Access ORM
db = patstat.orm()

# Importing the as models
from epo.tipdata.patstat.database.models import TLS222_APPLN_JP_CLASS

## APPLN_ID

In [2]:
# Import table TLS201
from epo.tipdata.patstat.database.models import TLS201_APPLN

show_join = db.query(
    TLS201_APPLN.appln_id,
    TLS201_APPLN.appln_auth,
    TLS222_APPLN_JP_CLASS.jp_class_scheme,
    TLS222_APPLN_JP_CLASS.jp_class_symbol
).join(
    TLS201_APPLN, TLS222_APPLN_JP_CLASS.appln_id == TLS201_APPLN.appln_id
).limit(1000)

show_join_df = patstat.df(show_join)
show_join_df

,appln_id,appln_auth,jp_class_scheme,jp_class_symbol
0,26796245,JP,FTERM,3C056/CA21
1,33090823,JP,FTERM,4H104/AA01Z
2,31382595,JP,FTERM,2H111/EA33
3,35529245,JP,FTERM,5L096/DA03
4,28843534,JP,FTERM,2H076/BB05
...,...,...,...,...
995,30185176,JP,FI,A63B 71/14 B
996,28967451,JP,FTERM,5J090/KA12
997,37400934,JP,FI,H10N10 /852
998,32252412,JP,FTERM,5D378/DE52


## JP_CLASS_SCHEME

This attribute indicates which of the two schemes are applied to the application: FI or FTERM. These classifications are being stored in DOCDB as supplied by the national office without inspection of the contents. The EPO does not hold any responsibility for content, format or validity.

In [3]:
from sqlalchemy import func

jp_schema = db.query(
    TLS222_APPLN_JP_CLASS.jp_class_scheme,
    func.count(TLS222_APPLN_JP_CLASS.appln_id).label('Number of applications')
).group_by(
    TLS222_APPLN_JP_CLASS.jp_class_scheme
).order_by(
    func.count(TLS222_APPLN_JP_CLASS.appln_id).desc()
)

jp_schema_df = patstat.df(jp_schema)
jp_schema_df

,jp_class_scheme,Number of applications
0,FTERM,331988545
1,FI,81887401


## JP_CLASS_SYMBOL

The two schemes FI and FTERM consist of symbols, which can be up to 50 characters long. Again, as for `jp_class_scheme`, these classifications are being stored in DOCDB as supplied by the national office without inspection of the contents. The EPO does not hold any responsibility for content, format or validity.

In [4]:
symb_appln = db.query(
    TLS222_APPLN_JP_CLASS.appln_id,
    func.count(TLS222_APPLN_JP_CLASS.jp_class_symbol).label('Number of symbols')
).group_by(
    TLS222_APPLN_JP_CLASS.appln_id
).having(
    func.count(TLS222_APPLN_JP_CLASS.jp_class_symbol) > 1  # Consider only applications with more than 1 class symbol
).order_by(
    func.count(TLS222_APPLN_JP_CLASS.jp_class_symbol).desc()
).limit(1000)

symb_appln_df = patstat.df(symb_appln)
symb_appln_df

,appln_id,Number of symbols
0,521423091,534
1,471404782,452
2,514424628,451
3,457197560,448
4,474957640,448
...,...,...
995,36021186,263
996,582973440,263
997,29465260,263
998,375184314,263


In [5]:
from epo.tipdata.patstat.database.models import TLS202_APPLN_TITLE

symb_appln = db.query(
    TLS222_APPLN_JP_CLASS.appln_id,
    TLS201_APPLN.appln_nr,
    TLS201_APPLN.appln_auth,
    TLS222_APPLN_JP_CLASS.jp_class_symbol,
    TLS222_APPLN_JP_CLASS.jp_class_scheme,
    TLS202_APPLN_TITLE.appln_title
).join(
    TLS201_APPLN, TLS222_APPLN_JP_CLASS.appln_id == TLS201_APPLN.appln_id
).join(
    TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id
).filter(
    TLS222_APPLN_JP_CLASS.appln_id == 471404782
).order_by(
    TLS222_APPLN_JP_CLASS.jp_class_scheme
)

symb_appln_df = patstat.df(symb_appln)
symb_appln_df

,appln_id,appln_nr,appln_auth,jp_class_symbol,jp_class_scheme,appln_title
0,471404782,2016059925,JP,H01L27 /105 441,FI,SEMICONDUCTOR DEVICE AND MANUFACTURING METHOD ...
1,471404782,2016059925,JP,H01L27 /08 102E,FI,SEMICONDUCTOR DEVICE AND MANUFACTURING METHOD ...
2,471404782,2016059925,JP,H10K59 /12,FI,SEMICONDUCTOR DEVICE AND MANUFACTURING METHOD ...
3,471404782,2016059925,JP,H01L21 /31 C,FI,SEMICONDUCTOR DEVICE AND MANUFACTURING METHOD ...
4,471404782,2016059925,JP,H01L27 /108 671C,FI,SEMICONDUCTOR DEVICE AND MANUFACTURING METHOD ...
...,...,...,...,...,...,...
447,471404782,2016059925,JP,4M104/DD35,FTERM,SEMICONDUCTOR DEVICE AND MANUFACTURING METHOD ...
448,471404782,2016059925,JP,4M104/DD18,FTERM,SEMICONDUCTOR DEVICE AND MANUFACTURING METHOD ...
449,471404782,2016059925,JP,5F083/GA21,FTERM,SEMICONDUCTOR DEVICE AND MANUFACTURING METHOD ...
450,471404782,2016059925,JP,4M104/BB05,FTERM,SEMICONDUCTOR DEVICE AND MANUFACTURING METHOD ...


In [6]:
appln_symb = db.query(
    TLS222_APPLN_JP_CLASS.jp_class_symbol,
    func.count(TLS222_APPLN_JP_CLASS.appln_id).label('Number of symbols')
).group_by(
    TLS222_APPLN_JP_CLASS.jp_class_symbol
).having(
    func.count(TLS222_APPLN_JP_CLASS.appln_id) > 1  # Consider only applications with more than 1 class symbol
).order_by(
    func.count(TLS222_APPLN_JP_CLASS.appln_id).desc()
)

appln_symb_df = patstat.df(appln_symb)
appln_symb_df

,jp_class_symbol,Number of symbols
0,4C086/AA01,160578
1,4C086/AA02,153584
2,5K067/EE02,147576
3,4C086/NA14,129221
4,4F100/BA03,127776
...,...,...
986652,A61K35 /56 AFC,2
986653,B24B 9/10 B,2
986654,H04L 5/04,2
986655,B25B 1/18 A,2
